# Explore MLHD+, Canonical Data Dump, and MusicBrainz formats

Stage 1 goal: look at the raw files directly before writing any loader code in
`src/songpsychic/data/`, so the parser matches what's actually in the files instead of
assumptions from the docs.

Run first:
- `python scripts/fetch_sample_data.py` (one MLHD+ partial shard ~2GB, one complete shard ~15GB)
- `python scripts/download_data.py --dataset canonical` (ID/name lookup table, small)
- `python scripts/download_data.py --dataset musicbrainz --files mbdump.tar.bz2 mbdump-derived.tar.bz2` (~8GB)

In [2]:
import tarfile
from pathlib import Path

import pandas as pd

RAW = Path("../data/raw")

## MLHD+ shards

Each shard is a `.tar` of per-user files. Start by just listing what's inside — don't assume a
naming scheme or column layout yet.

In [7]:
partial_path = RAW / "mlhd" / "mlhdplus-partial-0.tar"

with tarfile.open(partial_path) as tar:
    partial_members = tar.getmembers()

# Tars also contain directory entries; extractfile() returns None for those, so keep regular files only
partial_files = [m for m in partial_members if m.isfile()]

print(f"{len(partial_members)} members ({len(partial_files)} files)")
for m in partial_members[:10]:
    print(m.name, m.size, "file" if m.isfile() else "dir/other")

37358 members (37342 files)
00 0 dir/other
00/004927e0-092d-4eea-b928-eea7a8ca3b48.txt.zst 60367 file
00/00174fbf-5e8e-461c-b2ad-95f9319a8fcf.txt.zst 55183 file
00/0004fbd1-1673-497f-bf3d-4a40e2acadd7.txt.zst 165096 file
00/0082cb10-4630-4148-9342-f22c4b3f6d2b.txt.zst 55778 file
00/0002f253-70c5-4fd0-9578-ab3d8df6c5a4.txt.zst 18772 file
00/0026de04-67f6-4f80-aa5e-5dd504453a82.txt.zst 79043 file
00/00721df7-61fe-4329-86b2-6d6c401b4b95.txt.zst 48900 file
00/0052102e-c1ab-4e7f-aa16-efa6720b7926.txt.zst 27575 file
00/00d8548f-fd8a-400d-8601-4a0ca35a89b5.txt.zst 26323 file


In [8]:
# Raw bytes of the first file — figure out delimiter, columns, header (if any), before parsing
with tarfile.open(partial_path) as tar:
    with tar.extractfile(partial_files[0]) as f:
        raw = f.read(2000)

print(raw.decode("utf-8", errors="replace"))

���uuB�/���w���^����7�]Gߚng?���0����wg/�/�Q2� u����h���b!� �5���וֹP|A!u��Pa�£%w�,T!A��a��Ci�%�e@AӰ o�go���"E)��0�L�4��6/dؒ��+��ES_ ���,��ts���ޣ;�Ȱ�	�vB0<��	K�nͮ�Ԑr��\12�k(�?��da��M�� �n��AB��0�Q""�� +>��QUu-�V1�yz�<E�pӹN��:�g��,�����I[��k!�i�Jq���g1C�f�ȩ��dZh�t�!swʮG�5���=� C=BRj21�Oh4
B�	{V\�G]@���ϬY�d�Lr�/ɪiO<��m?��=���,š}*7�3T�X����)E� ��U����N��Hzt���r����R��*�#&�y*���^ ]�9��|+f
�����p$����҈�!vjG�=�����x�p�y�t���(E�� ��+B��Ȫ��s%^im�A��
�������J��dcښ1��WA�W��k�&����x�P`^S�V`]TFȘ��QT�]��\(( �$%V���*�B�ٯ�(o�m�� Sڭ�����(��c5Fꥄ �.��t�JW�]Q�C���je�vT��{�\G'�:"_oi��ܻQA!�;G��,O��)�P�Y7�P)�.RLVh�X�A�G���z- a��nܑ�yL��fRP�f� �)�SE�ST������*����՜����ʑ�7��<��K��B :m B�r��zʖ�2�L\ H2���@��<�ʢ��8u0�$�y
i�h�N"P`�� C�)�@�=x醬�|�؈a\�ssM�a8@֭f��4f��,��C %�~�m�j�|�Z8v�3|O���DA�)����b�j!�	aw���:Aq@AQ������/LBLų�Ɯ����fǽ�*B;�*��,�h)F��b�-�eGXb@y����U�����5�\$�P˝L�dg�z�g�AE1 ��^V;

## Compare against the complete shard

"Partial" vs "complete" is an MLHD+ distinction (per the docs, complete users have more total
listens) — check whether the file *format* is actually the same, only the content differs.

In [9]:
complete_path = RAW / "mlhd" / "mlhdplus-complete-0.tar"

with tarfile.open(complete_path) as tar:
    complete_members = tar.getmembers()
    complete_files = [m for m in complete_members if m.isfile()]
    with tar.extractfile(complete_files[0]) as f:
        complete_raw = f.read(2000)

print(f"{len(complete_members)} members ({len(complete_files)} files)")
print(complete_raw.decode("utf-8", errors="replace"))

37358 members (37342 files)
B-a�i�I��v[�N�vǳo5�λa���e�>	�CHC��ŷ[��Ij�%L���y$aR��&�+#	��!I��ǐDh��!�z�c�Kfzb�K��<����,�9��8��8q7VZE��c�����}��n%��H������|I�BP>�W_�(��E�N�U��������މj�,�Iu A���jP�彰����N�[�|����qU�mT�<�Ą�QjA�h=�c�XD���/Q� ��ϴP�"�\j�ă�L���h2������qa�=�:!���'�����c�{Zĳ�Z\A��E�O�A<P��#|t'�|r�}��:�Cj�ɍd�����&D�Z�c�sx�zq�>�'�K�]b�jd�,�I8�6����pb�̧[
�1��0�����1V�c�8�� C$��
�
�in-��'�6dN{LX����~DߢJ�h#Nh�ӆs�I��&e��J;Y,}���<9�+���M�+�x�8-�NX�A��j�T�:,=�d�T���[
�8����X��О��Κ�%���I���5>,�p=�$�%����I�u��K<����Ĭ!ClK���ˢ(��(u�z�:Xl複H��H���`�rvh�-��w���WI,�;t��|�ۉZ��F���M���e-��x�a��U���F��-�sb:F짫��Z$:(��F�I�a�6�:&H�h)'׃��lw* ��+�S�ti�p?���nR�����0-Ɖ�pk�c�������J��!)�����:�-�U�&75�sWҀ�r'ZV�d�^�.��������Y���͊�8qk����+F��A��]&�����l�


## Canonical Data Dump

Needs `python scripts/download_data.py --dataset canonical` first. The archive is `.tar.zst`;
decompressing it needs the `zstandard` package (`pip install zstandard`).

In [5]:
import zstandard

canonical_dir = RAW / "canonical"
archive = next(canonical_dir.glob("*.tar.zst"))
extracted_dir = canonical_dir / "extracted"

with open(archive, "rb") as fh:
    dctx = zstandard.ZstdDecompressor()
    with dctx.stream_reader(fh) as reader:
        with tarfile.open(fileobj=reader, mode="r|") as tar:
            tar.extractall(extracted_dir)

csv_paths = list(extracted_dir.rglob("*.csv"))
csv_paths

/var/folders/7h/yrt1yr854r12vf79mcwbjpx80000gn/T/ipykernel_3687/4133251730.py:11: DeprecationWarning: Python 3.14 will, by default, filter extracted tar archives and reject files or modify their metadata. Use the filter argument to control this behavior.
  tar.extractall(extracted_dir)


[PosixPath('../data/raw/canonical/extracted/musicbrainz-canonical-dump-20261017-080003/canonical/canonical_recording_redirect.csv'),
 PosixPath('../data/raw/canonical/extracted/musicbrainz-canonical-dump-20261017-080003/canonical/canonical_musicbrainz_data.csv'),
 PosixPath('../data/raw/canonical/extracted/musicbrainz-canonical-dump-20261017-080003/canonical/canonical_release_redirect.csv')]

In [6]:
for csv_path in csv_paths:
    print("---", csv_path.name, "---")
    display(pd.read_csv(csv_path, nrows=5))

--- canonical_recording_redirect.csv ---


,recording_mbid,canonical_recording_mbid,canonical_release_mbid
0,b80391be-803d-4ee6-bcce-837479a0fc0a,ecb125d7-d23e-4d76-8282-745713563110,ffb4aba3-8aa4-479d-9a62-1bbe881804b8
1,3bd7729a-aafa-4ea1-863a-824c6391b596,ecb125d7-d23e-4d76-8282-745713563110,ffb4aba3-8aa4-479d-9a62-1bbe881804b8
2,22760537-0a77-41d7-88ec-0e935e58677a,ecb125d7-d23e-4d76-8282-745713563110,ffb4aba3-8aa4-479d-9a62-1bbe881804b8
3,c46d2015-cc39-4632-8b32-fa14d8372d65,ecb125d7-d23e-4d76-8282-745713563110,ffb4aba3-8aa4-479d-9a62-1bbe881804b8
4,81dd4e68-ad37-4f7c-809e-a13d83ff9159,ecb125d7-d23e-4d76-8282-745713563110,ffb4aba3-8aa4-479d-9a62-1bbe881804b8


--- canonical_musicbrainz_data.csv ---


,id,artist_credit_id,artist_mbids,artist_credit_name,release_mbid,release_name,recording_mbid,recording_name,combined_lookup,score
0,1,1,89ad4ac3-39f7-470e-963a-56509c546377,Various Artists,432d5329-fdbb-4803-9f74-ca2d71118ab1,Songs of Quigley's Village Volume 2,5c29abbd-b602-4f22-8c40-bc119abc86c9,Bedtime Prayer,variousartistsbedtimeprayer,87910
1,2,1,89ad4ac3-39f7-470e-963a-56509c546377,Various Artists,432d5329-fdbb-4803-9f74-ca2d71118ab1,Songs of Quigley's Village Volume 2,c6839c82-bf0a-43a5-80ce-25d2b7f973f7,Quigley’s Village Theme,variousartistsquigleysvillagetheme,87910
2,3,1,89ad4ac3-39f7-470e-963a-56509c546377,Various Artists,61efbf58-5be4-40ba-9bef-49c92cc5b8ca,Sofé Linzy,7b1f193b-c9ba-48d6-bd6f-8afd02d489a2,Pot Pourri Sega,variousartistspotpourrisega,188739
3,4,1,89ad4ac3-39f7-470e-963a-56509c546377,Various Artists,82b37a32-e983-4b6b-b353-f38b9018ea7f,滾石民歌時代百大經典,4cbd4a88-eff8-47e5-8d18-683b1cdff891,乡愁四韵,variousartistsXiang Chou Si Yun,191119
4,5,1,89ad4ac3-39f7-470e-963a-56509c546377,Various Artists,82b37a32-e983-4b6b-b353-f38b9018ea7f,滾石民歌時代百大經典,77c52b21-0ce0-4c49-a99d-d25ebd4f5f71,白霏霏,variousartistsBai Fei Fei,191119


--- canonical_release_redirect.csv ---


,release_mbid,canonical_release_mbid,release_group_mbid
0,b3577744-e721-4357-bc60-d79c02863dac,b3577744-e721-4357-bc60-d79c02863dac,e8bee759-9efc-35c2-93d7-09ace9123467
1,96ccdd6e-ef20-32b2-ac99-8840c1a191c1,3a25785e-50ec-383c-89cf-fd512181449c,8b6f133a-2fdf-3cc2-b84d-1c889adc0939
2,1a1befee-6d0c-4d09-a150-788010080911,3a25785e-50ec-383c-89cf-fd512181449c,8b6f133a-2fdf-3cc2-b84d-1c889adc0939
3,9af8d45c-d4df-310d-baff-e5b7d956c2c8,3a25785e-50ec-383c-89cf-fd512181449c,8b6f133a-2fdf-3cc2-b84d-1c889adc0939
4,a0dc813b-ff05-4e47-b825-7cf8311e5052,3a25785e-50ec-383c-89cf-fd512181449c,8b6f133a-2fdf-3cc2-b84d-1c889adc0939


## MusicBrainz database dump (core + derived)

Needs `python scripts/download_data.py --dataset musicbrainz --files mbdump.tar.bz2
mbdump-derived.tar.bz2` first. This is PostgreSQL's plain-text `COPY` format: one file per table,
tab-separated, `\N` for null, **no header row and no column names** — those live in the
MusicBrainz server schema, not in the dump itself.

In [ ]:
mb_core_path = RAW / "musicbrainz" / "mbdump.tar.bz2"

with tarfile.open(mb_core_path, mode="r:bz2") as tar:
    mb_core_members = tar.getmembers()

print(f"{len(mb_core_members)} members")
for m in mb_core_members[:20]:
    print(m.name, m.size)

In [ ]:
# Peek the `artist` table specifically — found by name, not assumed path
artist_member = next(m for m in mb_core_members if m.name.endswith("/artist"))

with tarfile.open(mb_core_path, mode="r:bz2") as tar:
    with tar.extractfile(artist_member) as f:
        artist_raw = f.read(2000)

print(artist_raw.decode("utf-8", errors="replace"))

In [ ]:
first_row = artist_raw.decode("utf-8", errors="replace").splitlines()[0]
print(f"{first_row.count(chr(9)) + 1} tab-separated columns")
print(first_row)

## Derived dump (tags, ratings, search indexes)

Same `COPY` format as the core dump, but licensed CC BY-NC-SA (vs. CC0 for the core dump) — keep
the two separate if this data is ever reused outside this project.

In [ ]:
mb_derived_path = RAW / "musicbrainz" / "mbdump-derived.tar.bz2"

with tarfile.open(mb_derived_path, mode="r:bz2") as tar:
    mb_derived_members = tar.getmembers()

print(f"{len(mb_derived_members)} members")
for m in mb_derived_members[:20]:
    print(m.name, m.size)

No column names ship with either dump — table schemas live in the MusicBrainz server repo's SQL:
https://github.com/metabrainz/musicbrainz-server/blob/master/admin/sql/CreateTables.sql.
Needed before writing a real loader in `src/songpsychic/data/`.

## Notes

Fill in once the cells above have run:
- Per-listen format in an MLHD+ member file (columns, delimiter, units for the timestamp):
- Row/member count for the partial shard vs. the complete shard:
- Which canonical CSV column joins to which MLHD+ field:
- Which MusicBrainz tables (by file name) are actually needed for stage 3 (artist, tag/genre
  relationships), vs. which can be skipped:
- Anything surprising (missing values, encoding issues, duplicate rows):